In [1]:
import numpy as np
import pandas as pd

def gerar_dados_xenoverse_10_colunas(n=3000, semente=42):
    rng = np.random.default_rng(semente)

    # Id
    id_patrulheiro = [f"PT-{i:04d}" for i in range(1, n + 1)]

    # Nível do personagem (1 a 199)
    nivel = rng.normal(70, 20, n).clip(1, 199).round().astype(int)

    # Tempo de jogo em horas
    tempo_jogo_horas = (nivel * rng.uniform(1.5, 3.5, n) + rng.normal(0, 15, n)).clip(5, 500).round(1)

    # Medalhas PT
    medalhas_pt = rng.lognormal(mean=6.2, sigma=0.6, size=n)
    idx_outlier = rng.choice(n, size=15, replace=False)
    medalhas_pt[idx_outlier] *= rng.uniform(5, 12, size=15)
    medalhas_pt = medalhas_pt.round().astype(int)

    # Raças
    racas = np.array(["Saiyan", "Human", "Majin", "Namekian", "Frieza"])
    raca = rng.choice(racas, size=n, p=[0.45, 0.25, 0.12, 0.08, 0.10])

    # Taxa de sucesso em missões (%)
    quest_sucess = (20 + 0.4 * nivel + rng.normal(0, 8, n)).clip(20, 100).round(2)

    # Ki Máximo
    ki_maximo = (3 + (nivel / 20) + rng.normal(0, 0.5, n)).clip(3, 10).round().astype(int) * 100

    # Stamina Máxima
    stamina_maxima = (3 + (nivel / 25) + rng.normal(0, 0.5, n)).clip(3, 10).round().astype(int) * 100

    # Mentor Atual
    mentores = np.array(["krillin", "tien", "yamcha", "piccolo", "raditz", "gohan (kid)", "nappa", "vegeta", "zarbon", "dodoria", "captain ginyu", "frieza", "cooler", "android 18", "android 16", "cell", "lord slug", "majin buu", "hercule", "gohan (adult) and videl", "gotenks", "turles", "broly", "beerus", "whis", "pan", "jaco", "goku", "gohan (future)","bardock", "hit", "bojack", "zamasu", "nenhum",])
    p_raw = np.arange(len(mentores), 0, -1)
    p = p_raw / np.sum(p_raw)
    mentor_atual = rng.choice(mentores, size=n, p=p)

    # Criando o DataFrame Limpo
    df_limpo = pd.DataFrame({
        "id_patrulheiro": id_patrulheiro,
        "nivel": nivel,
        "tempo_jogo_horas": tempo_jogo_horas,
        "medalhas_pt": medalhas_pt,
        "raca": raca,
        "quest_sucess": quest_sucess,
        "ki_maximo": ki_maximo,
        "stamina_maxima": stamina_maxima,
        "mentor_atual": mentor_atual
    })

    # 2. Criando a versão suja
    df_sujo = df_limpo.copy()

    # Inserindo valores nulos (NaN) em várias colunas
    mask_nivel = rng.uniform(0, 1, n) < 0.05
    mask_medalhas = rng.uniform(0, 1, n) < 0.08
    mask_raca = rng.uniform(0, 1, n) < 0.04
    mask_ki = rng.uniform(0, 1, n) < 0.03
    mask_stamina = rng.uniform(0, 1, n) < 0.03
    mask_mentor = rng.uniform(0, 1, n) < 0.02

    df_sujo.loc[mask_nivel, "nivel"] = np.nan
    df_sujo.loc[mask_medalhas, "medalhas_pt"] = np.nan
    df_sujo.loc[mask_mentor, "mentor_atual"] = np.nan
    df_sujo.loc[mask_ki, "ki_maximo"] = np.nan
    df_sujo.loc[mask_stamina, "stamina_maxima"] = np.nan

    # Inserindo níveis inválidos (negativos ou acima de 199) em ~3% dos dados válidos
    idx_nivel_ruim = rng.choice(n, size=int(n * 0.03), replace=False)
    valores_invalidos_nivel = rng.choice([-5, -1, 0, 250, 300, 999], size=len(idx_nivel_ruim))
    df_sujo.loc[idx_nivel_ruim, "nivel"] = valores_invalidos_nivel

    # Inserindo medalhas negativas em alguns registros
    idx_medalhas_negativas = rng.choice(n, size=20, replace=False)
    df_sujo.loc[idx_medalhas_negativas, "medalhas_pt"] = rng.integers(-500, -1, size=20)

    # --- SUJEIRA EM STRINGS (Espaços aleatórios e formatação) ---
    def sujar_espacos(texto):
        if pd.isna(texto):
            return texto
        espacos_inicio = " " * rng.integers(1, 4) if rng.random() > 0.5 else ""
        espacos_fim = " " * rng.integers(1, 4) if rng.random() > 0.5 else ""
        return f"{espacos_inicio}{texto}{espacos_fim}"

    mask_txt = rng.uniform(0, 1, n) < 0.25
    df_sujo.loc[mask_txt, "raca"] = df_sujo.loc[mask_txt, "raca"].apply(sujar_espacos)
    mask_txt_mentor = rng.uniform(0, 1, n) < 0.20
    df_sujo.loc[mask_txt_mentor, "mentor_atual"] = df_sujo.loc[mask_txt_mentor, "mentor_atual"].apply(sujar_espacos)

    # --- DUPLICAÇÃO DE LINHAS ---
    # Escolhe aleatoriamente por volta de 3% a 5% de linhas para duplicar
    qtd_duplicadas = int(n * 0.04)
    linhas_para_duplicar = df_sujo.sample(n=qtd_duplicadas, random_state=rng.integers(0, 10000))
    df_sujo = pd.concat([df_sujo, linhas_para_duplicar], ignore_index=True)

    # Salvando os arquivos em CSV
    df_limpo.to_csv("DBXV2_limpo.csv", index=False)
    df_sujo.to_csv("DBXV2_sujo.csv", index=False)

    print(f"Total limpo: {n} registros | Total sujo (com duplicadas): {len(df_sujo)} registros | Total de colunas: {df_sujo.shape[1]}")
    return df_limpo, df_sujo

# Executando a função
df_limpo, df_sujo = gerar_dados_xenoverse_10_colunas()

Total de registros: 3000 | Total de colunas: 9
